<a href="https://colab.research.google.com/github/shahanasyedismail/CSA1731-AI/blob/main/decision_tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
from collections import Counter


def entropy(rows):
    counts = Counter(r[-1] for r in rows)
    total = len(rows)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def info_gain(rows, col):
    total = len(rows)
    groups = {}
    for r in rows:
        groups.setdefault(r[col], []).append(r)
    remainder = sum(len(g) / total * entropy(g) for g in groups.values())
    return entropy(rows) - remainder


def majority(rows):
    return Counter(r[-1] for r in rows).most_common(1)[0][0]


def build_tree(rows, features, max_depth=None, depth=0):
    labels = set(r[-1] for r in rows)
    if len(labels) == 1:
        return labels.pop()                      # pure node
    if not features or (max_depth is not None and depth >= max_depth):
        return majority(rows)                    # no features left / depth limit

    best = max(features, key=lambda f: info_gain(rows, f))
    node = {"feature": best, "branches": {}, "default": majority(rows)}
    remaining = [f for f in features if f != best]

    for value in set(r[best] for r in rows):
        subset = [r for r in rows if r[best] == value]
        node["branches"][value] = build_tree(subset, remaining, max_depth, depth + 1)
    return node


def predict(tree, sample):
    while isinstance(tree, dict):
        value = sample[tree["feature"]]
        if value not in tree["branches"]:
            return tree["default"]               # unseen value
        tree = tree["branches"][value]
    return tree


def print_tree(tree, names, indent=""):
    if not isinstance(tree, dict):
        print(indent + "->", tree)
        return
    for value, sub in tree["branches"].items():
        print(f"{indent}[{names[tree['feature']]} = {value}]")
        print_tree(sub, names, indent + "    ")


if __name__ == "__main__":
    names = ["Outlook", "Temp", "Humidity", "Wind"]
    # last column = label (Play)
    data = [
        ["Sunny", "Hot", "High", "Weak", "No"],
        ["Sunny", "Hot", "High", "Strong", "No"],
        ["Overcast", "Hot", "High", "Weak", "Yes"],
        ["Rain", "Mild", "High", "Weak", "Yes"],
        ["Rain", "Cool", "Normal", "Weak", "Yes"],
        ["Rain", "Cool", "Normal", "Strong", "No"],
        ["Overcast", "Cool", "Normal", "Strong", "Yes"],
        ["Sunny", "Mild", "High", "Weak", "No"],
        ["Sunny", "Cool", "Normal", "Weak", "Yes"],
        ["Rain", "Mild", "Normal", "Weak", "Yes"],
        ["Sunny", "Mild", "Normal", "Strong", "Yes"],
        ["Overcast", "Mild", "High", "Strong", "Yes"],
        ["Overcast", "Hot", "Normal", "Weak", "Yes"],
        ["Rain", "Mild", "High", "Strong", "No"],
    ]

    print("Information gain per feature:")
    for i, n in enumerate(names):
        print(f"  {n}: {info_gain(data, i):.3f}")

    tree = build_tree(data, list(range(len(names))))
    print("\nDecision tree:")
    print_tree(tree, names)

    test = ["Sunny", "Cool", "High", "Strong"]
    print("\nPrediction for", test, "->", predict(tree, test))

Information gain per feature:
  Outlook: 0.247
  Temp: 0.029
  Humidity: 0.152
  Wind: 0.048

Decision tree:
[Outlook = Overcast]
    -> Yes
[Outlook = Rain]
    [Wind = Strong]
        -> No
    [Wind = Weak]
        -> Yes
[Outlook = Sunny]
    [Humidity = High]
        -> No
    [Humidity = Normal]
        -> Yes

Prediction for ['Sunny', 'Cool', 'High', 'Strong'] -> No
